# PSI Data Preprocessing

This notebook cleans the enriched publication dataset and produces `papers_filtered.csv` for downstream graph loading and analysis.

## Workflow

1. Load the raw `papers_topics.csv` export.
2. Use `updated_topics` as the source topic field, convert it into a clean semicolon-separated string, and export it as `topics` in the final file while retaining papers that have no updated topic labels yet.
3. Parse NC State author entries and derive helper fields.
4. Run quality checks before exporting the filtered file.


## Environment Check

Use the project virtual environment or another environment with the dependencies from `requirements.txt` installed before running the notebook.


In [1]:
# Select the venv kernel in VS Code (Python: env/bin/python)
import sys
print(sys.executable)

/Users/dharani/Desktop/PSI/env/bin/python


In [2]:
%pip install -r "../requirements.txt"


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Load The Raw Dataset

Read the enriched publication export and inspect the first few rows to confirm the schema.


In [3]:
import pandas as pd

papers = pd.read_csv(
    '../data/papers_topics.csv'
)

papers.head()

,Unnamed: 0,title,authors,nc_state_people,DOI,PMID,year,url,topics,abstract,semantically_matched_keywords_hf_0_6,semantically_matched_topics_hf_0_6,updated_topics,semantically_matched_topics_hf_0_8
0,0,Mapping variability of soil water content and ...,"Sayde, Chadi; Buelga, Javier Benitez; Rodrigue...",Chadi Sayde (csayde),10.1002/2013wr014983,NaN,2014,https://openalex.org/W2160181936,Soil Moisture and Remote Sensing; Thermal Anal...,Abstract The Actively Heated Fiber Optic (AHFO...,"['breeding', 'pest mgmt', 'outreach', 'microbe...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['soil', 'sensing']","[0, 28]"
1,1,Natural gas price uncertainty and the cost‐eff...,"Kern, Jordan D.; Characklis, Gregory W.; Foste...",Jordan Kern (jkern),10.1002/2014wr016533,NaN,2015,https://openalex.org/W2093917128,Water resources management and optimization; A...,Abstract Prolonged periods of low reservoir in...,"['breeding', 'pest mgmt', 'outreach', 'microbe...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",['risk management'],[53]
2,2,High-resolution wind speed measurements using ...,"Sayde, Chadi; Thomas, Christoph K.; Wagner, Ja...",Chadi Sayde (csayde),10.1002/2015gl066729,NaN,2015,https://openalex.org/W2258664150,Seismic Waves and Analysis; Meteorological Phe...,Abstract We present a novel technique to simul...,"['breeding', 'pest mgmt', 'outreach', 'microbe...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",['sensing'],[0]
3,3,Calibration of soil moisture sensing with subs...,"Benítez-Buelga, Javier; Rodríguez-Sinobas, Leo...",Chadi Sayde (csayde),10.1002/2015wr017897,NaN,2016,https://openalex.org/W2298265415,Soil Moisture and Remote Sensing; Soil and Uns...,Abstract The heat pulse probe method can be im...,"['breeding', 'pest mgmt', 'outreach', 'microbe...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['materials engineering', 'soil', 'modeling', ...","[0, 48, 5, 28, 45]"
4,4,Mapping high-resolution soil moisture and prop...,NaN,Chadi Sayde (csayde),10.1002/2016wr019031,NaN,2016,https://openalex.org/W2521661497,Soil Moisture and Remote Sensing; Soil and Uns...,This study demonstrated a new method for mappi...,"['breeding', 'pest mgmt', 'outreach', 'microbe...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[],[]


## Remove Unused Columns And Null Rows

These steps drop fields that are not needed in the current pipeline, remove the legacy raw `topics` column, convert `updated_topics` into a semicolon-separated text field, and prepare it to be saved as `topics` in the final output before name parsing begins. Papers are retained even when `updated_topics` is empty.


In [4]:
import ast


def normalize_updated_topics(value):
    if pd.isna(value):
        return None

    if isinstance(value, list):
        items = value
    else:
        raw = str(value).strip()
        if not raw or raw == '[]':
            return None

        try:
            parsed = ast.literal_eval(raw)
            items = parsed if isinstance(parsed, list) else [raw]
        except (ValueError, SyntaxError):
            items = [part.strip() for part in raw.split(';') if part.strip()]

    cleaned_items = [str(item).strip() for item in items if str(item).strip()]
    return '; '.join(cleaned_items) if cleaned_items else None


papers = papers.drop(columns=[
    'Unnamed: 0',
    'PMID',
    'url',
    'abstract',
    'topics',
    'semantically_matched_keywords_hf_0_6',
    'semantically_matched_topics_hf_0_6',
    'semantically_matched_topics_hf_0_8'
])

papers['updated_topics'] = papers['updated_topics'].apply(normalize_updated_topics)
papers = papers.rename(columns={'updated_topics': 'topics'})

In [5]:
required_columns = ['title', 'authors', 'DOI', 'year']
papers = papers.dropna(subset=required_columns)

In [6]:
papers.isna().sum()

title               0
authors             0
nc_state_people     0
DOI                 0
year                0
topics             71
dtype: int64

In [7]:
papers.shape

(3170, 6)

In [8]:
papers = papers[lambda x: x["year"].between(2016, 2025)]
papers.shape

(2743, 6)

## Split Author Fields

The raw author columns are converted into list-like structures so we can compare the full publication author list with the NC State-specific entries.

This prepares the data for downstream parsing, where each NC State entry is handled one person at a time.


In [9]:
authors = papers['authors'].str.split(';')

In [10]:
nc_authors_list = papers['nc_state_people'].str.split(';').apply(
    lambda x: [item.strip() for item in x] if x is not None else None
)

In [11]:
nc_authors_list.isna().sum()

np.int64(0)

## Parse NC State Entries

This section is where we turn the `nc_state_people` text field into structured identity information.

What happens here:

- each entry is parsed from the `Name (unityid)` format,
- names are normalized into a consistent author-style format,
- Unity IDs are extracted and stored separately,
- and `nc_authors` is built using a canonical name per Unity ID.

Using Unity ID as the identity anchor helps with an important edge case: the same person may appear under different last names or slightly different display names across records. When that happens, we keep one canonical author name for that Unity ID so the same NC person is not treated as multiple different people downstream.


In [12]:
import re
from collections import Counter, defaultdict

ENTRY_PATTERN = re.compile(r"^\s*(.+?)\s*\((.*?)\)\s*$")


def parse_nc_entry(entry):
    entry = str(entry).strip()
    if not entry:
        return None, None, True

    match = ENTRY_PATTERN.match(entry)
    if not match:
        # Keep the raw text as fallback name for author formatting.
        return entry, None, True

    name = match.group(1).strip()
    unity_id = match.group(2).strip().lower()

    # Empty unity IDs are invalid format for later filtering,
    # but we still keep the name for nc_authors generation.
    if not unity_id:
        return name, None, True

    return name, unity_id, False


def to_author_format(name):
    """
    Convert name to author style: 'Last, First Middle'.
    If already in comma format, keep it normalized.
    """
    raw = str(name).strip()
    if not raw:
        return ""

    if ',' in raw:
        last, given = raw.split(',', 1)
        return f"{last.strip()}, {' '.join(given.split()).strip()}".strip().rstrip(',')

    parts = [p for p in raw.split() if p]
    if len(parts) == 1:
        return parts[0]

    last = parts[-1]
    given = ' '.join(parts[:-1])
    return f"{last}, {given}".strip().rstrip(',')


def choose_canonical_name(name_counts):
    ranked_names = sorted(
        name_counts.items(),
        key=lambda item: (-item[1], -len(item[0]), item[0].lower())
    )
    return ranked_names[0][0] if ranked_names else None


def build_unity_id_name_map(series):
    unity_id_to_names = defaultdict(Counter)

    for value in series.dropna():
        for entry in str(value).split(';'):
            entry = entry.strip()
            if not entry:
                continue

            name, unity_id, _ = parse_nc_entry(entry)
            formatted_name = to_author_format(name) if name else None
            if unity_id and formatted_name:
                unity_id_to_names[unity_id][formatted_name] += 1

    return {
        unity_id: choose_canonical_name(name_counts)
        for unity_id, name_counts in unity_id_to_names.items()
    }


UNITY_ID_TO_CANONICAL_NAME = build_unity_id_name_map(papers['nc_state_people'])


def extract_nc_authors_and_ids(row):
    if pd.isna(row['nc_state_people']):
        return pd.Series([None, None])

    nc_entries = [
        x.strip()
        for x in str(row['nc_state_people']).split(';')
        if x.strip()
    ]

    nc_authors = []
    unity_ids = []

    for entry in nc_entries:
        nc_name, unity_id, _ = parse_nc_entry(entry)
        if nc_name:
            formatted = to_author_format(nc_name)
            canonical_name = UNITY_ID_TO_CANONICAL_NAME.get(unity_id, formatted) if unity_id else formatted
            if canonical_name:
                nc_authors.append(canonical_name)

        if unity_id:
            unity_ids.append(unity_id)

    return pd.Series([
        '; '.join(nc_authors) if nc_authors else None,
        '; '.join(unity_ids) if unity_ids else None
    ])


papers[['nc_authors', 'unity_ids']] = papers.apply(
    extract_nc_authors_and_ids,
    axis=1
)


In [13]:
papers.head()

,title,authors,nc_state_people,DOI,year,topics,nc_authors,unity_ids
3,Calibration of soil moisture sensing with subs...,"Benítez-Buelga, Javier; Rodríguez-Sinobas, Leo...",Chadi Sayde (csayde),10.1002/2015wr017897,2016,materials engineering; soil; modeling; carbon ...,"Sayde, Chadi",csayde
5,Temporal variability in the importance of hydr...,"Nelson, Natalie G.; Muñoz-Carpena, Rafael; Nea...",Natalie Nelson (nnelson4),10.1002/2016WR020196,2017,NaN,"Nelson, Natalie",nnelson4
6,Failure of Taylor's hypothesis in the atmosphe...,"Cheng, Yu; Sayde, Chadi; Li, Qi; Basara, Jeffr...",Chadi Sayde (csayde),10.1002/2017gl073499,2017,sensing,"Sayde, Chadi",csayde
7,Spatial Patterns of Development Drive Water Use,"Sanchez, G. M.; Smith, J. W.; Terando, A.; Sun...",Georgina Sanchez (gmsanche); Ross Meentemeyer ...,10.1002/2017wr021730,2018,conservation; social sciences; resource econom...,"Sanchez, Georgina; Meentemeyer, Ross",gmsanche; rkmeente
9,A Proteomic-Based Quantitative Analysis of the...,"Wang, Jack P.; Tunlaya-Anukit, Sermsawat; Shi,...",Jack Wang (jpwang); Joel Ducoste (jducoste); F...,10.1002/9781118883303.ch4,2016,breeding; pest mgmt; outreach; microbes; Agtec...,"Wang, Jack; Ducoste, Joel; Isik, Fikret",jpwang; jducoste; fisik


## Quality Checks

These checks verify that the NC parsing step stayed internally consistent.

Specifically, we confirm that:

- the number of parsed NC authors matches the number of NC State entries,
- the number of extracted Unity IDs stays aligned with those parsed authors,
- invalid `nc_state_people` formats are flagged,
- and any Unity IDs associated with multiple name variants can be reviewed.

This makes it easier to catch malformed rows and also gives visibility into cases where one researcher may appear under more than one name.


In [14]:
def count_items(value):
    if pd.isna(value) or value is None:
        return 0
    return len([x for x in str(value).split(';') if x.strip()])


def has_invalid_nc_state_people_format(value):
    if pd.isna(value) or value is None:
        return False

    for entry in str(value).split(';'):
        entry = entry.strip()
        if not entry:
            continue

        name, unity_id, is_invalid = parse_nc_entry(entry)
        if is_invalid:
            return True

    return False


papers['count_nc_state_people'] = papers['nc_state_people'].apply(count_items)
papers['count_nc_authors'] = papers['nc_authors'].apply(count_items)
papers['count_unity_ids'] = papers['unity_ids'].apply(count_items)

papers['has_invalid_nc_state_people_format'] = papers['nc_state_people'].apply(has_invalid_nc_state_people_format)

papers['counts_match'] = (
    (papers['count_nc_state_people'] == papers['count_nc_authors']) &
    (papers['count_nc_state_people'] == papers['count_unity_ids'])
)

mismatch_rows = papers[(~papers['has_invalid_nc_state_people_format']) & (papers['counts_match'] == False)]
invalid_format_rows = papers[papers['has_invalid_nc_state_people_format']]

if len(mismatch_rows) > 0:
    print(f"❌ MISMATCH FOUND (excluding invalid nc_state_people format rows) : {len(mismatch_rows)}")
    display(mismatch_rows[
        ['title',
         'authors',
         'nc_state_people',
         'nc_authors',
         'unity_ids',
         'count_nc_state_people',
         'count_nc_authors',
         'count_unity_ids']
    ])
else:
    print("✅ All valid-format rows match correctly.")

unity_id_to_names = defaultdict(Counter)
for value in papers['nc_state_people'].dropna():
    for entry in str(value).split(';'):
        entry = entry.strip()
        if not entry:
            continue

        name, unity_id, _ = parse_nc_entry(entry)
        formatted_name = to_author_format(name) if name else None
        if unity_id and formatted_name:
            unity_id_to_names[unity_id][formatted_name] += 1

name_variant_rows = [
    {
        'unity_id': unity_id,
        'canonical_name': choose_canonical_name(name_counts),
        'raw_name_variants': '; '.join(
            f"{name} ({count})" for name, count in sorted(name_counts.items(), key=lambda item: (-item[1], item[0].lower()))
        ),
        'variant_count': len(name_counts)
    }
    for unity_id, name_counts in unity_id_to_names.items()
    if len(name_counts) > 1
]

if name_variant_rows:
    print(f"ℹ️ Found {len(name_variant_rows)} Unity IDs with multiple name variants. Canonical names will be used in nc_authors.")
    display(pd.DataFrame(name_variant_rows).sort_values(['variant_count', 'unity_id'], ascending=[False, True]).head(20))
else:
    print("✅ No multi-name Unity ID variants found in the current dataset.")


✅ All valid-format rows match correctly.
✅ No multi-name Unity ID variants found in the current dataset.


In [15]:
rows_to_drop_mask = (~papers['has_invalid_nc_state_people_format']) & (~papers['counts_match'])

dropped_rows = papers[rows_to_drop_mask].copy()
papers_filtered = papers[~rows_to_drop_mask].copy().reset_index(drop=True)

print(f"Rows dropped: {len(dropped_rows)}")
print(f"Rows remaining: {len(papers_filtered)}")

Rows dropped: 0
Rows remaining: 2743


## Export The Filtered Dataset

Drop temporary QC columns and write the final `papers_filtered.csv` file used by the Neo4j loader. The exported file uses the cleaned topic values under the standard `topics` column name.


In [16]:
import csv


def format_csv_value(value, force_quote=False):
    if pd.isna(value):
        return ''

    text = str(value)
    needs_quote = force_quote or any(char in text for char in [',', '"', '\n'])
    if '"' in text:
        text = text.replace('"', '""')
    return f'"{text}"' if needs_quote else text


# Drop QC helper columns from final CSV
qc_helper_cols = [
    'count_nc_state_people',
    'count_nc_authors',
    'count_unity_ids',
    'has_invalid_nc_state_people_format',
    'counts_match'
]
papers_filtered = papers_filtered.drop(columns=qc_helper_cols)

# Save filtered dataframe to CSV with the topics column always quoted
output_path = "../data/papers_filtered.csv"
columns = papers_filtered.columns.tolist()

with open(output_path, 'w', encoding='utf-8', newline='') as handle:
    handle.write(','.join(columns) + '\n')
    for _, row in papers_filtered.iterrows():
        formatted_row = [
            format_csv_value(row[column], force_quote=(column == 'topics'))
            for column in columns
        ]
        handle.write(','.join(formatted_row) + '\n')

print(f"Saved papers_filtered.csv with {len(papers_filtered)} rows and {papers_filtered.shape[1]} columns")


Saved papers_filtered.csv with 2743 rows and 8 columns


## Quick Validation

This final check compares unique NC author counts across the derived and original NC State columns after export.


In [17]:

df = pd.read_csv("../data/papers_filtered.csv", usecols=["nc_authors", "nc_state_people"])

# 1) Unique from nc_authors
unique_nc_authors = (
    df["nc_authors"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
)
unique_nc_authors = unique_nc_authors[unique_nc_authors.ne("")].nunique()

# 2) Unique names from nc_state_people (remove trailing "(unityid)")
unique_nc_state_people = (
    df["nc_state_people"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
    .str.replace(r"\s*\([^)]*\)\s*$", "", regex=True)  # strip (unity_id)
    .str.strip()
)
unique_nc_state_people = unique_nc_state_people[unique_nc_state_people.ne("")].nunique()

print("Unique nc_authors:", unique_nc_authors)
print("Unique nc_state_people names:", unique_nc_state_people)


Unique nc_authors: 1615
Unique nc_state_people names: 1615
